# Chapter 20 - Diffusion from Scratch (live)

Companion notebook for [Chapter 20](../part-6-frontier/20-diffusion-multimodal.md). We build a full DDPM on 2D data: a fixed **forward** process that noises a spiral into a Gaussian, a tiny **denoiser** trained to predict the noise, and a **reverse** sampler that turns pure noise back into the spiral - with the denoising **trajectory** rendered frame by frame.

**What you'll see:** structure dissolving into noise, a falling training loss, samples reconstructing the data manifold, and the step-by-step reverse trajectory.

> Pure NumPy + matplotlib - the denoiser is a from-scratch MLP with manual backprop. No PyTorch needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = np.cumprod(alphas)            # cumulative product -> closed-form noising

## 1. The data: a 2D spiral

In [ ]:
def make_spiral(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.sqrt(rng.random(n)) * 3 * np.pi
    r = t / (3 * np.pi)
    x = np.stack([r * np.cos(t), r * np.sin(t)], axis=1)
    return (x / x.std()).astype(np.float64)

data = make_spiral()
plt.figure(figsize=(4.2, 4.2))
plt.scatter(data[:, 0], data[:, 1], s=3, alpha=0.5)
plt.title('Target distribution'); plt.gca().set_aspect('equal'); plt.tight_layout(); plt.show()

## 2. Forward process: noise it to a Gaussian

The closed form lets us jump to any noise level: $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$. As $t$ grows the signal vanishes and the data becomes $\mathcal N(0, I)$ - which is *why* sampling can start from pure noise.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, t in zip(axes, [0, 25, 75, 150, 199]):
    eps = np.random.randn(*data.shape)
    xt = np.sqrt(abar[t]) * data + np.sqrt(1 - abar[t]) * eps
    ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5)
    ax.set_title(f't = {t}'); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Forward diffusion: spiral -> isotropic Gaussian'); plt.tight_layout(); plt.show()

## 3. The denoiser: predict the noise

A small MLP $\epsilon_\theta(x_t, t)$ takes a noised point plus a sinusoidal time embedding and predicts the noise that was added. Training is plain MSE regression - the whole reason diffusion is more stable than GANs.

In [ ]:
def time_embed(t, dim=16):
    t = np.asarray(t, dtype=np.float64)[:, None] / T
    freqs = (2.0 ** np.arange(dim // 2))[None] * np.pi
    a = t * freqs
    return np.concatenate([np.sin(a), np.cos(a)], axis=1)

TDIM, H, IN = 16, 128, 2 + 16
def init(seed=0):
    rng = np.random.default_rng(seed)
    s = lambda a, b: rng.standard_normal((a, b)) * np.sqrt(1.0 / a)
    return {'W0': s(IN, H), 'b0': np.zeros(H), 'W1': s(H, H), 'b1': np.zeros(H),
            'W2': s(H, 2), 'b2': np.zeros(2)}

def net_forward(p, xt, temb):
    inp = np.concatenate([xt, temb], axis=1)
    z0 = inp @ p['W0'] + p['b0']; a0 = np.tanh(z0)
    z1 = a0 @ p['W1'] + p['b1']; a1 = np.tanh(z1)
    pred = a1 @ p['W2'] + p['b2']
    return pred, (inp, a0, a1)

def net_backward(p, cache, dpred):
    inp, a0, a1 = cache
    g = {}
    g['W2'] = a1.T @ dpred; g['b2'] = dpred.sum(0)
    da1 = dpred @ p['W2'].T; dz1 = da1 * (1 - a1 ** 2)
    g['W1'] = a0.T @ dz1; g['b1'] = dz1.sum(0)
    da0 = dz1 @ p['W1'].T; dz0 = da0 * (1 - a0 ** 2)
    g['W0'] = inp.T @ dz0; g['b0'] = dz0.sum(0)
    return g

In [ ]:
class Adam:
    """Minimal Adam over a dict of numpy arrays (updated in place)."""
    def __init__(self, params, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
        self.p, self.lr, self.b1, self.b2, self.eps = params, lr, b1, b2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0
    def step(self, grads):
        self.t += 1
        for k in self.p:
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * grads[k]
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * grads[k] ** 2
            mh = self.m[k] / (1 - self.b1 ** self.t)
            vh = self.v[k] / (1 - self.b2 ** self.t)
            self.p[k] -= self.lr * mh / (np.sqrt(vh) + self.eps)

## 4. Train the denoiser

In [ ]:
params = init()
opt = Adam(params, lr=2e-3)
N = len(data); losses = []
for step in range(6000):
    idx = np.random.randint(0, N, 256)
    x0 = data[idx]
    t = np.random.randint(0, T, 256)
    eps = np.random.randn(256, 2)
    xt = np.sqrt(abar[t])[:, None] * x0 + np.sqrt(1 - abar[t])[:, None] * eps
    pred, cache = net_forward(params, xt, time_embed(t))
    diff = pred - eps
    losses.append((diff ** 2).mean())
    opt.step(net_backward(params, cache, 2 * diff / 256))

print('final loss: %.4f' % np.mean(losses[-200:]))
plt.figure(figsize=(5, 3))
plt.plot(np.convolve(losses, np.ones(50) / 50, 'valid'))
plt.title('Denoiser training loss (smoothed)'); plt.xlabel('step'); plt.ylabel('MSE')
plt.tight_layout(); plt.show()

## 5. Reverse process: sample from pure noise

In [ ]:
def sample(params, n=3000, keep=(199, 150, 100, 50, 20, 0)):
    x = np.random.randn(n, 2)
    frames = {}
    for t in reversed(range(T)):
        eps = net_forward(params, x, time_embed(np.full(n, t)))[0]
        mean = (x - betas[t] / np.sqrt(1 - abar[t]) * eps) / np.sqrt(alphas[t])
        x = mean + (np.sqrt(betas[t]) * np.random.randn(n, 2) if t > 0 else 0.0)
        if t in keep:
            frames[t] = x.copy()
    return x, frames

samples, frames = sample(params)
plt.figure(figsize=(4.2, 4.2))
plt.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5, color='crimson')
plt.title('Generated samples (from noise)'); plt.gca().set_aspect('equal')
plt.tight_layout(); plt.show()

## 6. The denoising trajectory

In [ ]:
order = [199, 150, 100, 50, 20, 0]
fig, axes = plt.subplots(1, len(order), figsize=(16, 3))
for ax, t in zip(axes, order):
    f = frames[t]
    ax.scatter(f[:, 0], f[:, 1], s=3, alpha=0.5, color='crimson')
    ax.set_title(f't = {t}'); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Reverse diffusion: noise -> spiral, one step at a time'); plt.tight_layout(); plt.show()

## Takeaway

Generation is hard, but **denoising is easy** - so diffusion decomposes the impossible task into hundreds of trivial regression steps. The forward process is fixed math; only the denoiser is learned; and sampling just runs the denoiser in reverse. Scale this denoiser to a U-Net/DiT over image latents and you have Stable Diffusion.

**Try it:** swap the spiral for two moons or a checkerboard; reduce the sampler to 20 steps (DDIM-style) and compare.

[Back to Chapter 20](../part-6-frontier/20-diffusion-multimodal.md) | [Solutions](../solutions/20-diffusion-multimodal-solutions.md)